# Indirect Standardization and the SMR: Comparing Mortality When Rates Are Too Fragile

*Demographic Analysis Series · Module 1 (Mortality) · Lesson 1.3a*
*Author: Jiajia Li | Published: September 26, 2026 | Reading time: about 25 minutes*

[Lesson 1.2b](/blog/demographic-m1l2b-standardization-worked-example) ended with a limitation: direct standardization needs a full age-specific rate schedule for every population in the comparison. For Malaysia and Australia, two national statistical systems, that requirement is easily met. For a mining town, an occupational cohort, or one birth cohort inside a single survey, it is not. This lesson develops the method built for exactly that situation: indirect standardization and the standardized mortality ratio (SMR).

By the end of this notebook you will be able to:

- Explain the expected-deaths idea: apply one trusted standard schedule to your population's age structure and compare observed with expected deaths
- Implement a validated `standardize_indirect()` in Python and unit-test it with the standard-population identity
- Judge, with a Poisson simulation, when small death counts force the indirect route
- Explain why each SMR carries its own implicit standard, and why a ratio of two SMRs is not itself a standardized ratio

## 1. The Problem: A Schedule Built on Handfuls of Deaths

The direct method divides deaths by exposure in every age group. In a national register each cell holds thousands of deaths, so the division is stable. In a small population the same computation divides handfuls of deaths by handfuls of person-years: a rate of 0 per 1,000 because nobody aged 25 to 29 happened to die this year, or 400 per 1,000 because two deaths landed in a cell with five residents. Standardizing such a schedule does not rescue it; the weighted average of nineteen noisy rates stays noisy.

Carmichael's Chapter 2 (p. 68) offers the indirect route, which never computes the study population's age-specific rates at all. Instead, borrow a rate schedule you trust, typically from a large national population, and apply it to the study population's age structure. That application yields the number of deaths you would *expect* if the standard's risks ruled in your population. Comparing the deaths actually observed with that expectation gives the standardized mortality ratio.

We continue with the 1988 Malaysia-Australia data from Lessons 1.1c through 1.2b, so the indirect answer can be checked against the direct answers we already have.

In [ ]:
# ---- 1. Load Tables 1.2/1.3 (same embedded data as Lessons 1.1c-1.2b) ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO
from IPython.display import display

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 11,
    'figure.dpi': 125,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'legend.frameon': False,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})
GREEN, RUST = '#35634e', '#a95b3d'

# Carmichael (2016), Ch. 1: Table 1.2 (rates per 1,000) and Table 1.3 (persons per 1,000,000), 1988.
rates_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,16.6,9.8,12.8,7.6
1-4,1.7,0.5,1.1,0.4
5-9,0.6,0.2,0.5,0.2
10-14,0.6,0.3,0.4,0.2
15-19,1.0,1.1,0.5,0.4
20-24,1.5,1.6,0.6,0.5
25-29,1.7,1.5,0.9,0.5
30-34,2.0,1.4,1.1,0.6
35-39,2.4,1.5,1.5,0.8
40-44,3.4,2.2,2.3,1.2
45-49,5.5,3.4,3.2,2.1
50-54,9.9,6.0,5.9,3.4
55-59,15.4,10.0,9.9,5.5
60-64,25.5,17.3,17.0,8.7
65-69,38.3,27.2,27.8,13.8
70-74,62.7,45.3,46.6,23.5
75-79,86.2,71.9,69.8,40.7
80-84,144.3,110.7,119.0,71.4
85+,173.4,186.6,142.8,147.7
"""
pop_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,14254,7597,13464,7245
1-4,56773,30324,53804,28954
5-9,61207,37762,58012,35803
10-14,56002,38712,53613,36758
15-19,52776,43626,50772,41743
20-24,50193,40771,49060,39297
25-29,42748,42914,44839,41982
30-34,35704,40234,38326,39933
35-39,30427,38778,31269,38391
40-44,23858,36248,23075,34537
45-49,20473,27923,19767,26357
50-54,17052,23855,17190,22798
55-59,12615,22708,13604,21924
60-64,10043,21764,10513,22336
65-69,7373,17701,8498,19973
70-74,4608,12857,5246,16200
75-79,3430,8712,3976,12555
80-84,1399,4471,1675,7827
85+,963,2319,1399,6111
"""
rates = pd.read_csv(StringIO(rates_csv), index_col='age')
pop = pd.read_csv(StringIO(pop_csv), index_col='age')
deaths = pop * rates / 1000

exposure = pd.DataFrame({
    'Malaysia': pop['m_malaysia'] + pop['f_malaysia'],
    'Australia': pop['m_australia'] + pop['f_australia'],
})
structure = exposure / exposure.sum()
asdr = pd.DataFrame({
    'Malaysia': (deaths['m_malaysia'] + deaths['f_malaysia']) / exposure['Malaysia'] * 1000,
    'Australia': (deaths['m_australia'] + deaths['f_australia']) / exposure['Australia'] * 1000,
})
total_deaths = pd.Series({
    'Malaysia': deaths[['m_malaysia', 'f_malaysia']].sum().sum(),
    'Australia': deaths[['m_australia', 'f_australia']].sum().sum(),
})
cdr = total_deaths / exposure.sum() * 1000

summary = pd.DataFrame({
    'Total deaths (per million pop.)': total_deaths.round(0),
    'CDR per 1,000': cdr.round(2),
})
display(summary)
assert np.allclose(cdr.to_numpy(), [4.952, 7.251], atol=1e-3)

## 2. The Expected-Deaths Idea

For study population $i$ and standard schedule $m_s$:

$$E_i = \sum_x m_s(x)\,P_i(x), \qquad \mathrm{SMR}_i = \frac{O_i}{E_i},$$

where $P_i(x)$ is $i$'s population in age group $x$, $O_i$ its total observed deaths, and $E_i$ the deaths expected had the standard's age-specific rates applied to $i$'s structure. An SMR of 1 means the population buried exactly as many people as the standard schedule predicts for a population of that shape; 1.5 means 50% more.

Notice what the calculation consumes: a *total* death count (robust, because it pools all ages), an *age distribution* (usually known from a census even when vital registration is weak), and *one external rate schedule* (robust, because it comes from a large population). Every fragile ingredient of the direct method is replaced by a sturdy one.

One subtlety is worth flagging now and dissecting in Section 4. Algebraically, $\mathrm{SMR}_i = D_i \big/ \sum_x m_s(x)\,P_i(x)$ is exactly the direct standardized ratio comparing $i$ with $s$ *using $i$'s own age structure as the standard*. Each SMR therefore sits on its own private standard.

In [ ]:
# ---- 2. A validated indirect-standardization function ----
def standardize_indirect(observed_deaths, population, std_rates, unit=1000):
    """SMR: observed deaths divided by deaths expected under a standard schedule.

    observed_deaths: total deaths in the study population (scalar).
    population: Series of study-population counts by age.
    std_rates: Series of standard age-specific rates per `unit` population,
        on the same age index as population.
    Returns (smr, expected_deaths); both are needed for honest reporting.
    """
    if not population.index.equals(std_rates.index):
        raise ValueError('population and standard rates must share an identical age index')
    if (std_rates < 0).any():
        raise ValueError('standard rates must be nonnegative')
    expected = float((population * std_rates / unit).sum())
    if expected <= 0:
        raise ValueError('expected deaths must be positive')
    return observed_deaths / expected, expected


# Toy check: two age groups, 1,000 persons each, standard rates 10 and 20 per 1,000.
# Expected deaths = 10 + 20 = 30; with 45 observed deaths the SMR must be 1.5.
toy_smr, toy_e = standardize_indirect(
    45, pd.Series([1000, 1000], index=['0-39', '40+']),
    pd.Series([10.0, 20.0], index=['0-39', '40+']))
assert np.isclose(toy_smr, 1.5) and np.isclose(toy_e, 30)

# Identity check: the standard population measured against its own schedule scores 1.
# The indirect analogue of the self-standardization identity in Lesson 1.2b.
smr_aus, e_aus = standardize_indirect(total_deaths['Australia'], exposure['Australia'], asdr['Australia'])
assert np.isclose(smr_aus, 1.0)

# The real calculation: Malaysia's deaths against the Australian schedule.
smr_mal, e_mal = standardize_indirect(total_deaths['Malaysia'], exposure['Malaysia'], asdr['Australia'])
print(f"Malaysia 1988: observed {total_deaths['Malaysia']:,.0f}, expected {e_mal:,.0f} "
      f"(per million population) -> SMR = {smr_mal:.3f}")

# Carmichael's Ch. 2 reports 1.52; the printed tables, rounded, give:
assert np.isclose(smr_mal, 1.52, atol=0.005)

# Algebraic identity from Section 2: the SMR must equal the direct standardized
# ratio that uses Malaysia's own structure as the standard (Lesson 1.2b).
direct_on_malaysia = cdr['Malaysia'] / (asdr['Australia'] * structure['Malaysia']).sum()
assert np.isclose(smr_mal, direct_on_malaysia)
print(f"Direct ratio on Malaysia's own structure (1.2b) = {direct_on_malaysia:.3f}  (same number)")

In [ ]:
# Figure 1: observed vs expected deaths by age, Malaysia under the Australian schedule
observed_age = deaths['m_malaysia'] + deaths['f_malaysia']
expected_age = exposure['Malaysia'] * asdr['Australia'] / 1000

fig, ax = plt.subplots(figsize=(9.4, 4.0), layout='constrained')
x = np.arange(len(observed_age))
width = 0.4
ax.bar(x - width / 2, observed_age, width, color=GREEN, label='Observed deaths')
ax.bar(x + width / 2, expected_age, width, color=RUST, alpha=0.75,
       label='Expected under Australian schedule')
ax.set_xticks(x)
ax.set_xticklabels(observed_age.index, rotation=45, ha='right')
ax.set_xlabel('Age group')
ax.set_ylabel('Deaths per million population')
ax.legend(loc='upper left')
ax.set_title("Where Malaysia's excess deaths sit: the young ages", loc='left', pad=14)
ax.set_axisbelow(True)
ax.grid(axis='y', color='#e6e8e3', linewidth=0.8)
plt.show()

print(f"Totals per million: observed {observed_age.sum():,.0f}, expected {expected_age.sum():,.0f}")

**Figure 1 | Observed vs expected deaths by age, Malaysia 1988, Australian schedule as standard.** Green bars count Malaysia's actual deaths per million population (Tables 1.2 and 1.3, both sexes combined). Rust bars apply Australia's age-specific death rates to Malaysia's age structure. The excess over expectation concentrates at young ages, where Malaysian rates run two to three times Australian ones; at 85+ observed deaths fall slightly below expectation, echoing the crossover of the two schedules plotted in Lesson 1.1c. The totals, 4,952 observed against 3,260 expected, give SMR = 1.52: Malaysia carries about 52% more mortality than the Australian schedule predicts for a population of Malaysia's young shape.

The expected-deaths construction also previews the trap of Section 4: "expected" is computed on Malaysia's structure, so this SMR silently uses Malaysia as its own standard population.

## 3. When Direct Fails: A Simulation

How small is too small for the direct method? Simulate it. Imagine districts whose age structure matches Malaysia's but whose true mortality follows the Australian schedule inflated by 50% at every age. The true comparative mortality against Australia is then exactly 1.5 under any standard: proportional schedules preserve any ranking, per Lesson 1.2a's theorem. Draw each district's deaths from independent Poisson distributions and estimate the comparison two ways:

- **Direct**: standardize the district's own noisy age-specific rates on Australia's structure and divide by Australia's crude rate.
- **Indirect**: divide observed total deaths by expected deaths under the Australian schedule.

Because the true rates are proportional, both estimators target the same 1.5; any difference in behavior is statistical noise, not a different question. The SMR's only random ingredient is the total count $O$, and under Poisson sampling $\mathrm{SE}(\mathrm{SMR}) \approx \mathrm{SMR}/\sqrt{O}$. The direct estimator carries nineteen independently noisy cells into the weighted sum.

In [ ]:
# ---- 3. Poisson simulation: direct vs indirect at small population sizes ----
rng = np.random.default_rng(20260926)  # fixed seed: lesson reproducibility

std_schedule = asdr['Australia']                # trusted standard, per 1,000
true_rates = 1.5 * std_schedule                 # district truth: 50% excess at every age
district_shape = structure['Malaysia']          # districts shaped like Malaysia
truth = 1.5
e_rate = float((district_shape * std_schedule / 1000).sum())  # expected-death rate per person


def simulate_district(n_persons, n_rep=400):
    pop_d = (district_shape * n_persons).to_numpy()
    lam = pop_d * true_rates.to_numpy() / 1000  # Poisson means by age
    d = rng.poisson(lam[None, :], size=(n_rep, len(lam)))     # (rep, age)
    smr_hat = d.sum(axis=1) / (pop_d * std_schedule.to_numpy() / 1000).sum()
    noisy = d / pop_d * 1000                    # district's own noisy ASDRs
    direct_hat = noisy @ structure['Australia'].to_numpy() / cdr['Australia']
    return pd.DataFrame({'SMR': smr_hat, 'Direct': direct_hat})


sizes = [2500, 5000, 10000, 25000, 100000]
sim = {n: simulate_district(n) for n in sizes}

rows = []
for n in sizes:
    s = sim[n]
    rows.append({'District pop.': f'{n:,}', 'E[observed deaths]': round(1.5 * n * e_rate, 1),
                 'SMR mean': round(s['SMR'].mean(), 3), 'SMR SD': round(s['SMR'].std(), 3),
                 'Direct mean': round(s['Direct'].mean(), 3), 'Direct SD': round(s['Direct'].std(), 3)})
display(pd.DataFrame(rows).set_index('District pop.'))

# Both estimators centered on the truth; direct is more dispersed at every size;
# the SMR's spread tracks its Poisson floor.
assert all(sim[n]['Direct'].std() > sim[n]['SMR'].std() for n in sizes)
assert abs(sim[100000]['SMR'].mean() - truth) < 0.05
assert abs(sim[100000]['Direct'].mean() - truth) < 0.05
assert abs(sim[100000]['SMR'].std() - truth / np.sqrt(1.5 * 100000 * e_rate)) < 0.02

In [ ]:
# Figure 2: estimator behavior at N = 5,000 (left); spread vs district size (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0), layout='constrained')

strip = sim[5000]
jit = rng.uniform(-0.14, 0.14, size=len(strip))
ax1.scatter(1 + jit, strip['SMR'], s=7, alpha=0.35, color=GREEN)
ax1.scatter(2 + jit, strip['Direct'], s=7, alpha=0.35, color=RUST)
ax1.axhline(truth, color='#7b8078', linewidth=1, linestyle='--')
ax1.text(2.42, truth, 'truth = 1.5', va='center', fontsize=9, color='#7b8078')
clip = (strip['Direct'] > 3.5).mean() * 100
ax1.text(0.97, 0.04, f'{clip:.0f}% of direct estimates lie above this window',
         transform=ax1.transAxes, ha='right', fontsize=8.5, color='#7b8078')
ax1.set_xlim(0.4, 2.9)
ax1.set_xticks([1, 2])
ax1.set_xticklabels(['Indirect (SMR)', 'Direct'])
ax1.set_ylim(0, 3.5)
ax1.set_ylabel('Estimated mortality ratio vs Australia')
ax1.set_title('400 districts of 5,000 people each', loc='left', pad=14)
ax1.set_axisbelow(True)
ax1.grid(axis='y', color='#e6e8e3', linewidth=0.8)

ax2.plot(sizes, [sim[n]['SMR'].std() for n in sizes], marker='o', markersize=4,
         color=GREEN, linewidth=1.6, label='Indirect (SMR)')
ax2.plot(sizes, [sim[n]['Direct'].std() for n in sizes], marker='s', markersize=4,
         color=RUST, linewidth=1.6, linestyle='--', label='Direct')
ax2.plot(sizes, [truth / np.sqrt(1.5 * n * e_rate) for n in sizes],
         color='#7b8078', linewidth=1, linestyle=':', label=r'SMR theory: $1.5/\sqrt{O}$')
ax2.set_xscale('log')
ax2.set_xticks(sizes)
ax2.set_xticklabels([f'{n:,}' for n in sizes])
ax2.set_xlabel('District population (log scale)')
ax2.set_ylabel('SD of 400 estimates')
ax2.legend(loc='upper right', fontsize=9)
ax2.set_title('Direct stays noisier at every size', loc='left', pad=14)
ax2.set_axisbelow(True)
ax2.grid(axis='y', color='#e6e8e3', linewidth=0.8)
plt.show()

**Figure 2 | Small populations punish the direct estimator.** Left: 400 simulated districts of 5,000 people, each with true mortality 1.5 times the Australian schedule at every age. Indirect SMRs (green) cluster around the truth; direct standardized ratios (rust) scatter far more widely, because each district's age-specific rates rest on a handful of deaths. Right: the standard deviation of each estimator against district size. The direct estimator is more dispersed at every size, and the SMR's spread tracks the Poisson benchmark $1.5/\sqrt{O}$ (dotted), its theoretical floor.

The indirect method wins here not by magic but by accounting: it spends the district's scarce information on one number, the total death count, and imports the age pattern from the standard. The price appears when the district's true age pattern of mortality differs from the standard's, which is exactly the comparison of Lesson 1.3b.

## 4. The Trap: A Ratio of SMRs Is Not an SMR

Section 2's identity says each SMR quietly standardizes on its own population's structure. Malaysia's SMR against the Australian schedule (1.52) equals the direct ratio built on Malaysia's young structure; turn the computation around and Australia's SMR against the Malaysian schedule (0.69) is the reciprocal of the direct ratio built on Australia's old structure. Both are internally valid answers to "how does this population compare with the external standard?"

But divide one by the other and the implied Malaysia-to-Australia ratio becomes 2.2, far above every like-for-like comparison from Lesson 1.2b (1.46 to 1.52). Nothing went wrong arithmetically; the two SMRs rest on different implicit standards, so their quotient compares Malaysia-on-Malaysia's-structure with Australia-on-Australia's-structure. The composition difference, the very thing standardization exists to remove, re-enters through the back door.

The practical rules, following Carmichael (pp. 68–71): an SMR validly compares one population against the external standard. SMRs computed for many groups are approximately comparable only when their age structures are similar, so discount small differences and never rank finely on them. When structures differ sharply, as with migrant or occupational populations, only a direct comparison on a common standard is defensible; if counts are too small for that, report the uncertainty rather than the ranking.

In [ ]:
# ---- 4. Each SMR carries its own implicit standard ----
smr_aus_vs_mal, _ = standardize_indirect(total_deaths['Australia'], exposure['Australia'], asdr['Malaysia'])

# Direct ratios on the two own-structure standards (recomputed from 1.2b's tables)
direct_on_aus = (asdr['Malaysia'] * structure['Australia']).sum() / cdr['Australia']
assert np.isclose(smr_aus_vs_mal, 1 / direct_on_aus)  # identity, other direction

smr_ratio = smr_mal / smr_aus_vs_mal
print(f"SMR Malaysia (Australian schedule) : {smr_mal:.3f}   implicit standard: Malaysia's structure")
print(f"SMR Australia (Malaysian schedule) : {smr_aus_vs_mal:.3f}   implicit standard: Australia's structure")
print(f"Ratio of the two SMRs              : {smr_ratio:.3f}")
print(f"Own-structure direct ratios        : {direct_on_aus:.3f} and {direct_on_malaysia:.3f}")

# The remaining standards from Lesson 1.2b, recomputed on the five-year grid
five_yr = [f'{x}-{x+4}' for x in range(0, 85, 5)] + ['85+']
segi = pd.Series([12, 10, 9, 9, 8, 8, 6, 6, 6, 6, 5, 4, 4, 3, 2, 1, 0.5, 0.5], index=five_yr)
who = pd.Series([8.86, 8.69, 8.60, 8.47, 8.22, 7.93, 7.61, 7.15, 6.59,
                 6.04, 5.37, 4.55, 3.72, 2.96, 2.21, 1.52, 0.91, 0.63], index=five_yr)


def to_five_yr(v):
    out = v.copy()
    out.loc['0-4'] = out.loc['0'] + out.loc['1-4']
    return out.loc[five_yr]


deaths5 = pd.DataFrame({
    'Malaysia': to_five_yr(deaths['m_malaysia'] + deaths['f_malaysia']),
    'Australia': to_five_yr(deaths['m_australia'] + deaths['f_australia'])})
asdr5 = deaths5 / pd.DataFrame({'Malaysia': to_five_yr(exposure['Malaysia']),
                                'Australia': to_five_yr(exposure['Australia'])}) * 1000

ratios = {'Australia': direct_on_aus, 'Malaysia': direct_on_malaysia}
for name, w in [('Segi', segi), ('WHO', who)]:
    wn = w / w.sum()
    ratios[name] = float((asdr5['Malaysia'] * wn).sum() / (asdr5['Australia'] * wn).sum())
pooled = (exposure['Malaysia'] + exposure['Australia']) / (exposure.sum().sum())
ratios['Pooled'] = float((asdr['Malaysia'] * pooled).sum() / (asdr['Australia'] * pooled).sum())

for k, v in ratios.items():
    print(f"  {k:10s} standard: {v:.3f}")
assert all(1.45 < v < 1.53 for v in ratios.values())  # the 1.2b stability band

In [ ]:
# Figure 3: every like-for-like ratio vs the ratio of SMRs
fig, ax = plt.subplots(figsize=(8.6, 3.4), layout='constrained')
names = list(ratios.keys())
vals = [ratios[k] for k in names]
ypos = np.arange(len(names)) + 1
ax.barh(ypos, vals, height=0.55, color=GREEN, alpha=0.85)
ax.barh(0, smr_ratio, height=0.55, color=RUST)
for y, v in zip(ypos, vals):
    ax.text(v + 0.02, y, f'{v:.2f}', va='center', fontsize=9, color=GREEN)
ax.text(smr_ratio + 0.02, 0, f'{smr_ratio:.2f}', va='center', fontsize=9, color=RUST)
ax.axvline(1.0, color='#7b8078', linewidth=1, linestyle='--')
ax.set_yticks(list(ypos) + [0])
ax.set_yticklabels([f'Direct, {k} standard' for k in names] + ['Ratio of two SMRs'])
ax.set_xlim(0, 2.6)
ax.set_xlabel('Malaysia-to-Australia mortality ratio')
ax.set_title('One of these comparisons is not like the others', loc='left', pad=14)
ax.set_axisbelow(True)
ax.grid(axis='x', color='#e6e8e3', linewidth=0.8)
plt.show()

**Figure 3 | The ratio of two SMRs contradicts every defensible comparison.** Green bars: the Malaysia-to-Australia mortality ratio under five common standards, each computed directly on one shared structure, all between 1.46 and 1.52. Rust bar: Malaysia's SMR divided by Australia's SMR, 2.21. Every green bar holds the standard fixed within the comparison; the rust bar mixes two different implicit standards and answers no well-posed question.

## 5. Check Your Understanding

1. A district records 86 deaths against 70 expected under the national schedule, SMR = 1.23. Using $\mathrm{SE}(\mathrm{SMR}) \approx \mathrm{SMR}/\sqrt{O}$, is the excess statistically distinguishable from 1?
2. Why should the standard schedule come from a large population? What happens to $E_i$ if the standard's own age-specific rates are noisy?
3. Two towns have SMRs of 1.4 and 1.5 against the same national schedule. Can you conclude that town B's mortality is about 7% higher than town A's?

**Answers.** (1) $\mathrm{SE} \approx 1.23/\sqrt{86} = 0.13$, so the SMR sits under two standard errors above 1 ($z \approx 1.7$). Treat it as suggestive, not established; the Poisson floor is unforgiving at this count. (2) The standard's rates become your benchmark: every death in $E_i$ inherits their noise, and the SMR's appeal rests on the benchmark being solid. A noisy standard reintroduces through $E_i$ the fragility you removed from $O_i$. (3) No. Each SMR standardizes on its own town's age structure. If the towns' structures differ, the two ratios rest on different implicit standards and their quotient answers nothing; you need both schedules on one common standard, direct if counts allow, before ranking them.

## Coming Next: Direct vs Indirect, and the Limits of Standardization

Lesson 1.3b puts the two methods side by side on the same data and asks what standardization cannot fix: dependence on an arbitrary standard, schedules that cross, and the cases where the honest answer is to report age-specific rates themselves.

[Back to the Demographic Analysis Series](/blog/series/demographic) · [Review Lesson 1.2b](/blog/demographic-m1l2b-standardization-worked-example)

## References and Data Sources

1. Carmichael, G. A. (2016). *Fundamentals of Demographic Analysis: Concepts, Measures and Methods*. Springer. Data: Ch. 1, Table 1.2 (age-specific death rates, per 1,000) and Table 1.3 (population by age and sex, per 1,000,000), Malaysia and Australia, 1988. Indirect standardization and the SMR: Ch. 2, pp. 68–71. [Springer book and chapter listing](https://link.springer.com/book/10.1007/978-3-319-23255-3).
2. Breslow, N. E., & Day, N. E. (1987). *Statistical Methods in Cancer Research, Volume II: The Design and Analysis of Cohort Studies*. IARC Scientific Publications No. 82. The canonical treatment of the SMR and its Poisson variability. [IARC publication page](https://publications.iarc.fr/Book-And-Report-Series/Iarc-Scientific-Publications/Statistical-Methods-In-Cancer-Research-Volume-II-The-Design-And-Analysis-Of-Cohort-Studies-1986).
3. Ahmad, O. B., Boschi-Pinto, C., Lopez, A. D., Murray, C. J. L., Lozano, R., & Inoue, M. (2001). *Age Standardization of Rates: A New WHO Standard*. GPE Discussion Paper Series No. 31. World Health Organization. Segi and WHO standard weights: Table 1, as transcribed in Lesson 1.2a. [Original WHO PDF](https://cdn.who.int/media/docs/default-source/gho-documents/global-health-estimates/gpe_discussion_paper_series_paper31_2001_age_standardization_rates.pdf).

**Reproducibility.** All data are embedded in the code cells, so execution requires no external downloads. Dependencies are NumPy, pandas, Matplotlib, and Jupyter/IPython. The simulation seed is fixed at 20260926. Assertions check the crude rates (4.95 and 7.25 per 1,000), the toy unit test, the standard-population identity (SMR = 1), the Malaysian SMR (1.52, matching Carmichael), both implicit-standard identities against Lesson 1.2b's direct ratios, and the simulation's centering and spread ordering.